# Module 03 — Notebook 02: Layer Conductance

## Learning Objectives
By completing this notebook, you will:
1. Apply `LayerConductance` to find which ResNet-50 layer contributes most to a prediction
2. Verify the completeness property across all layers (sum ≈ f(x) − f(baseline))
3. Visualize spatial conductance maps within convolutional layers
4. Compare conductance across multiple images and classes

## Prerequisites
- Module 03 Notebook 01 (GradCAM)
- Guide 02 (Conductance theory)

## Estimated Time: 15 minutes

---

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import urllib.request
import io
import json
from PIL import Image

from captum.attr import LayerConductance

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

In [ ]:
# Load ImageNet labels
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"

# Load dog image
DOG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
try:
    with urllib.request.urlopen(DOG_URL, timeout=10) as resp:
        raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    raw = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))

input_tensor = preprocess(raw).unsqueeze(0).to(DEVICE)
img_np = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0).numpy()

with torch.no_grad():
    logits = model(input_tensor)
    probs  = torch.softmax(logits, dim=1)
    top_class = probs.argmax().item()
    top_prob  = probs.max().item()

print(f"Prediction: {labels[top_class]} ({top_prob:.1%})")

# Zero baseline
baseline = torch.zeros_like(input_tensor)

## 1. Layer Conductance: Which Layer Decides?

`LayerConductance` computes Integrated Gradients at an intermediate layer. The **completeness property** guarantees:

$$\sum_i \text{Cond}_i^l = f(x) - f(x') \quad \text{for every layer } l$$

This allows direct comparison across layers — the layer with the highest total absolute conductance is the one doing the most "work" for this prediction.

In [ ]:
# Define the layers to analyze
# Each is the last block of each residual stage in ResNet-50
layers_to_analyze = {
    'layer1 (64ch, 56×56)':   model.layer1[-1],
    'layer2 (128ch, 28×28)':  model.layer2[-1],
    'layer3 (256ch, 14×14)':  model.layer3[-1],
    'layer4 (512ch, 7×7)':    model.layer4[-1],
}

N_STEPS = 50  # Integration steps

# Total prediction difference (theoretical target for each layer's sum)
with torch.no_grad():
    f_x    = logits[0, top_class].item()
    f_base = model(baseline)[0, top_class].item()
total_diff = f_x - f_base
print(f"f(x) - f(baseline) = {total_diff:.4f}")
print(f"(All layer conductance sums should ≈ {total_diff:.4f})\n")

# Compute conductance for each layer
conductance_results = {}
raw_attrs = {}

for name, layer in layers_to_analyze.items():
    lc = LayerConductance(model, layer)
    attr = lc.attribute(
        input_tensor,
        baselines=baseline,
        target=top_class,
        n_steps=N_STEPS
    )
    raw_attrs[name] = attr

    total_signed = attr.sum().item()
    total_abs    = attr.abs().sum().item()
    n_positive   = (attr > 0).float().mean().item()

    conductance_results[name] = {
        'signed':    total_signed,
        'absolute':  total_abs,
        'n_positive': n_positive,
    }
    print(f"{name[:30]:30s}  sum={total_signed:+.4f}  |sum|={total_abs:.4f}  "
          f"{n_positive:.0%} neurons positive")

In [ ]:
# Visualize: bar chart of total conductance per layer
layer_names  = [n.split(' ')[0] for n in conductance_results.keys()]
signed_vals  = [v['signed']   for v in conductance_results.values()]
abs_vals     = [v['absolute'] for v in conductance_results.values()]
pos_fraction = [v['n_positive'] for v in conductance_results.values()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'Layer Conductance — {labels[top_class]} (n_steps={N_STEPS})',
    fontsize=13
)

# Signed conductance per layer
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in signed_vals]
bars = axes[0].bar(layer_names, signed_vals, color=colors, alpha=0.85)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].axhline(total_diff, color='navy', linestyle='--', lw=1.5,
                 label=f'f(x)-f(x\')={total_diff:.3f}')
axes[0].set_ylabel('Total Signed Conductance')
axes[0].set_title('Signed Conductance by Layer')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, signed_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.002,
                  f'{val:+.3f}', ha='center', va='bottom', fontsize=8)

# Absolute conductance per layer
axes[1].bar(layer_names, abs_vals, color='steelblue', alpha=0.85)
axes[1].set_ylabel('Total Absolute Conductance')
axes[1].set_title('Absolute Conductance by Layer\n(activity level)')
for ax_col, val in enumerate(abs_vals):
    axes[1].text(ax_col, val + 0.002, f'{val:.3f}', ha='center', va='bottom',
                  fontsize=8)

# Fraction of neurons with positive conductance
axes[2].bar(layer_names, [p * 100 for p in pos_fraction],
             color='darkorange', alpha=0.85)
axes[2].axhline(50, color='black', linestyle='--', lw=0.8, label='50%')
axes[2].set_ylabel('% Neurons with Positive Conductance')
axes[2].set_title('Fraction of Neurons Supporting Prediction')
axes[2].set_ylim(0, 100)
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Completeness verification
print("\nCompleteness verification (signed conductance sum vs f(x)-f(x')):")
print(f"  Theoretical:   {total_diff:.5f}")
for name, vals in conductance_results.items():
    delta = abs(vals['signed'] - total_diff)
    status = 'OK' if delta < 0.1 else 'WARNING'
    print(f"  {name.split(' ')[0]:8s}: {vals['signed']:+.5f}  (error = {delta:.5f}) [{status}]")

## 2. Spatial Conductance Maps

For convolutional layers, the conductance output has spatial dimensions `(1, channels, H, W)`. Aggregating across channels gives a spatial map showing **which spatial locations** in the feature map contributed most to the prediction.

In [ ]:
import torch.nn.functional as F

def get_spatial_map(attr_tensor, size=(224, 224)):
    """
    Aggregate conductance across channels and upsample to image resolution.

    Parameters
    ----------
    attr_tensor : Tensor  (1, C, H, W)
    size        : tuple   target spatial resolution for upsampling

    Returns
    -------
    ndarray (size[0], size[1])  normalized to [0, 1]
    """
    # Sum absolute conductance across channels
    spatial = attr_tensor.abs().sum(dim=1)  # (1, H, W)
    # Upsample
    spatial_up = F.interpolate(
        spatial.unsqueeze(0) if spatial.dim() == 3 else spatial,
        size=size, mode='bilinear', align_corners=False
    ).squeeze()
    # Normalize to [0, 1]
    mn, mx = spatial_up.min(), spatial_up.max()
    return ((spatial_up - mn) / (mx - mn + 1e-8)).detach().cpu().numpy()


# Compute spatial maps for all layers
spatial_maps = {}
for name in raw_attrs:
    spatial_maps[name] = get_spatial_map(raw_attrs[name])

# Visualize: spatial conductance overlaid on input image
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle(
    f'Spatial Conductance Maps — {labels[top_class]}',
    fontsize=13
)

# Column 0: input image
for row in range(2):
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input Image', fontsize=10)
    axes[row, 0].axis('off')

# Columns 1–4: spatial maps
for col, (name, smap) in enumerate(spatial_maps.items(), start=1):
    short = name.split(' ')[0]
    dims  = name.split('(')[-1].rstrip(')')

    # Row 0: pure conductance heatmap
    im = axes[0, col].imshow(smap, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(f'{short}\n({dims})', fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: overlay
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(smap, alpha=0.55, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'{short} overlay', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print("""
Spatial conductance progression:
- layer1: diffuse — many neurons across the spatial grid contribute
- layer2: slightly concentrated
- layer3: more focused on object-relevant regions
- layer4: most concentrated on the classified object region
""")

## 3. Channel Conductance: Which Features Matter?

Besides spatial maps, we can look at **which channels** (feature detectors) have the highest conductance. High-conductance channels are the feature detectors most responsible for this prediction.

In [ ]:
# Analyze channel-level conductance for layer4
attr_layer4 = raw_attrs['layer4 (512ch, 7×7)']
# attr_layer4: (1, 2048, 7, 7)

# Mean absolute conductance per channel (across spatial dimensions)
channel_cond = attr_layer4.abs().mean(dim=(-2, -1)).squeeze(0).detach().cpu().numpy()
# channel_cond: (2048,)

# Top-20 channels by absolute conductance
top20_idx = np.argsort(channel_cond)[::-1][:20]
top20_vals = channel_cond[top20_idx]

# Bottom-20 channels (least important)
bot20_idx = np.argsort(channel_cond)[:20]
bot20_vals = channel_cond[bot20_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'Layer4 Channel Conductance — {labels[top_class]}',
    fontsize=12
)

# Full distribution
axes[0].hist(channel_cond, bins=100, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Mean Absolute Conductance')
axes[0].set_ylabel('Number of Channels')
axes[0].set_title('Channel Conductance Distribution\n(layer4, 2048 channels)')
axes[0].axvline(np.percentile(channel_cond, 90), color='red', linestyle='--',
                 label='90th percentile')
axes[0].axvline(np.percentile(channel_cond, 99), color='darkred', linestyle='--',
                 label='99th percentile')
axes[0].legend(fontsize=9)

# Top-20 channels
axes[1].barh(
    [f'ch {i}' for i in top20_idx], top20_vals[::-1],
    color='#e74c3c', alpha=0.85
)
axes[1].set_xlabel('Mean |Conductance|')
axes[1].set_title('Top-20 Channels by Conductance')

# Cumulative conductance: how many channels are needed?
sorted_cond = np.sort(channel_cond)[::-1]
cumulative = np.cumsum(sorted_cond) / sorted_cond.sum() * 100
axes[2].plot(range(1, len(cumulative) + 1), cumulative, color='navy', lw=2)
axes[2].axhline(80, color='red', linestyle='--', label='80% threshold')
axes[2].axhline(95, color='darkred', linestyle='--', label='95% threshold')
n_for_80 = np.searchsorted(cumulative, 80) + 1
n_for_95 = np.searchsorted(cumulative, 95) + 1
axes[2].axvline(n_for_80, color='red', linestyle=':', alpha=0.7)
axes[2].axvline(n_for_95, color='darkred', linestyle=':', alpha=0.7)
axes[2].set_xlabel('Number of Channels (ranked)')
axes[2].set_ylabel('Cumulative % of Total Conductance')
axes[2].set_title('How Many Channels Needed?')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

total_channels = len(channel_cond)
print(f"Layer4 has {total_channels} channels.")
print(f"  Top {n_for_80} channels ({n_for_80/total_channels:.1%}) account for 80% of conductance")
print(f"  Top {n_for_95} channels ({n_for_95/total_channels:.1%}) account for 95% of conductance")
print(f"  This means {total_channels-n_for_95} channels are nearly dormant for this prediction.")

## 4. Conductance Across Multiple Images

Is the "layer4 is most important" finding general, or image-specific? We run Layer Conductance on three images and compare the layer importance profile.

In [ ]:
# Load a second image (cat) for comparison
CAT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"
BIRD_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"

def load_and_preprocess(url):
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            img = Image.open(io.BytesIO(resp.read())).convert('RGB')
    except Exception:
        img = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))
    tensor = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        cls = model(tensor).argmax().item()
    return tensor, cls

cat_tensor, cat_class   = load_and_preprocess(CAT_URL)
bird_tensor, bird_class = load_and_preprocess(BIRD_URL)

print(f"Cat prediction:  {labels[cat_class]}")
print(f"Bird prediction: {labels[bird_class]}")

images_to_compare = [
    (input_tensor, top_class,  'Dog'),
    (cat_tensor,  cat_class,   'Cat'),
    (bird_tensor, bird_class,  'Bird/Insect'),
]

layer_names_short = ['layer1', 'layer2', 'layer3', 'layer4']
layer_modules = [model.layer1[-1], model.layer2[-1],
                  model.layer3[-1], model.layer4[-1]]

multi_results = {}  # image_label → {layer_name → signed_sum}

print("\nComputing layer conductance across 3 images...")
for tensor, cls, img_label in images_to_compare:
    baseline_i = torch.zeros_like(tensor)
    multi_results[img_label] = {}
    for lname, lmodule in zip(layer_names_short, layer_modules):
        lc = LayerConductance(model, lmodule)
        attr = lc.attribute(
            tensor, baselines=baseline_i,
            target=cls, n_steps=30  # Fewer steps for speed
        )
        multi_results[img_label][lname] = attr.abs().sum().item()
    print(f"  {img_label}: {multi_results[img_label]}")

# Normalize each image's results to sum to 1 (percentage of total activity)
for img_label in multi_results:
    total = sum(multi_results[img_label].values())
    for lname in multi_results[img_label]:
        multi_results[img_label][lname] /= total

In [ ]:
# Grouped bar chart: layer importance by image
x = np.arange(len(layer_names_short))
width = 0.25
img_colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(12, 6))

for i, (img_label, color) in enumerate(
        zip(multi_results.keys(), img_colors)):
    vals = [multi_results[img_label][l] * 100
            for l in layer_names_short]
    bars = ax.bar(x + i * width, vals, width,
                   label=img_label, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(layer_names_short)
ax.set_ylabel('% of Total Layer Conductance')
ax.set_title(
    'Layer Conductance Profile Across Different Images\n'
    '(Normalized absolute conductance)',
    fontsize=12
)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Which layer dominates for each image?
print("Dominant layer per image:")
for img_label, result in multi_results.items():
    dominant = max(result, key=result.get)
    pct = result[dominant] * 100
    pred_cls = [cls for t, cls, lbl in images_to_compare if lbl == img_label][0]
    print(f"  {img_label:12s} → {dominant} ({pct:.0f}% of total) | Pred: {labels[pred_cls][:25]}")

## 5. Exercise: Conductance at Different n_steps

Layer Conductance uses the IG integral, which requires numerical approximation with `n_steps`. Assess how convergence quality changes with `n_steps`.

**Task:** For the dog image, compute Layer Conductance at `layer4` with `n_steps` ∈ {10, 25, 50, 100}. Plot how the total conductance sum (and its deviation from `f(x) - f(baseline)`) changes.

In [ ]:
# Exercise: Layer Conductance convergence with n_steps

step_counts = [10, 25, 50, 100]
convergence_sums = []

lc_layer4 = LayerConductance(model, model.layer4[-1])

for n in step_counts:
    attr = lc_layer4.attribute(
        input_tensor,
        baselines=baseline,
        target=top_class,
        n_steps=n
    )
    total_cond = attr.sum().item()
    convergence_sums.append(total_cond)
    error = abs(total_cond - total_diff)
    print(f"n_steps={n:4d}: sum={total_cond:+.5f}  |error|={error:.5f}")

# Plot convergence
errors = [abs(s - total_diff) for s in convergence_sums]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Layer Conductance Convergence vs n_steps (layer4)', fontsize=12)

axes[0].plot(step_counts, convergence_sums, 'o-', color='steelblue',
              linewidth=2, markersize=8)
axes[0].axhline(total_diff, color='red', linestyle='--',
                  label=f'Target: f(x)-f(x\')={total_diff:.4f}')
axes[0].set_xlabel('n_steps')
axes[0].set_ylabel('Sum of Layer4 Conductance')
axes[0].set_title('Conductance Sum vs n_steps')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(step_counts, errors, 'o-', color='darkorange',
                   linewidth=2, markersize=8)
axes[1].axhline(0.05, color='red', linestyle='--', label='|error|=0.05 (marginal)')
axes[1].axhline(0.01, color='green', linestyle='--', label='|error|=0.01 (good)')
axes[1].set_xlabel('n_steps')
axes[1].set_ylabel('|Approximation Error| (log scale)')
axes[1].set_title('Approximation Error vs n_steps')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find the first n_steps where error < 0.05
good_steps = next((n for n, e in zip(step_counts, errors) if e < 0.05), None)
print(f"\nFirst n_steps with |error| < 0.05: {good_steps}")
print(f"Recommendation: use n_steps={good_steps or 50} for this model/layer combination.")

# Auto-check
assert errors[-1] < errors[0], \
    "Error should decrease as n_steps increases"
print("\nCheck passed: error decreases with more integration steps.")

## Summary

### What You Learned

1. **Layer Conductance** applies the IG integral at intermediate layers; the result satisfies completeness for every layer independently
2. **Comparing layers** by their total signed conductance identifies the "decision-making" layer — typically layer4 for ResNet-50 on ImageNet
3. **Spatial conductance maps** show which image regions are active in each layer (layer4 most concentrated on the object)
4. **Channel analysis** reveals that a small fraction of channels accounts for most conductance — most neurons are dormant for any given prediction
5. **n_steps convergence** matters: at least n_steps=25–50 is needed for reliable layer conductance values

### Key API
```python
from captum.attr import LayerConductance

lc = LayerConductance(model, model.layer4[-1])
attr = lc.attribute(
    input_tensor, baselines=baseline,
    target=class_idx, n_steps=50
)
# attr: (1, channels, H, W)
# attr.sum().item() ≈ f(x) - f(baseline)  [completeness]
```

### What's Next

**Notebook 03:** Neuron Conductance — drill down to individual neurons to understand what specific feature detectors are responsible for each prediction.